# Laboratorio #7

* Josue Say - 228801
* Flavio Galán - 22386

## Repositorio

- [Enlace](https://github.com/JosueSay/labs-ds/tree/main/lab7)

Para este laboratorio se trabajó con los datos de **@traficogt** en data.

> **Nota:** Se utilizó python 3.12.6 para compatibilidad con librerias.

## Librerias

In [ ]:
import json, os
import pandas as pd
import numpy as np

# Limpieza lenguaje
import re, unicodedata
import spacy

nlp = spacy.load("es_core_news_sm")
STOP_ES = nlp.Defaults.stop_words

## Constantes


In [46]:
DATA_DIR = "data/"
DATA_FILE = DATA_DIR + "traficogt.txt"
DATA_CLEAN = DATA_DIR + "clean_traficogt.csv"

DATA_CLEAN_IN  = "data/clean_traficogt.csv"
DATA_CLEAN_OUT = "data/clean_traficogt_v2.csv"

DATA_USE = "data/clean_traficogt_v3"

In [47]:
URL_RE   = re.compile(r"https://\S+")
EMOJI_RE = re.compile(r"[\U0001F300-\U0001FAFF\u2600-\u26FF]+")

PUNCT_BASE = r"""[.,;:"'(){}\[\]\/\\\-–—_«»<>…]"""
PUNCT_KEPT = r"[!?¡¿]"

## Sección 1

In [48]:
def processTweetsToCsv(input_path="data/traficogt.txt", output_path="data/clean_traficogt.csv"):
    """
    Lee tweets en formato JSONL (uno por línea), extrae campos específicos
    y escribe un CSV. Si un campo no existe o viene como null, se escribe 'null'
    en el CSV (usando na_rep='null').

    Columnas:
      tweet_id, date, user_id, username, followers_count, friends_count, statuses_count,
      raw_content, reply_count, retweet_count, like_count, quote_count, conversation_id,
      hashtags, mentioned_users, mentioned_users_ids, view_count, place, coordinates,
      in_reply_to_tweet_id, in_reply_to_user_id, in_reply_to_username,
      has_quote, quoted_tweet_id, quoted_user_id, quoted_username
    """

    # Detectar codificación simple (UTF-16 o UTF-8/UTF-8 BOM)
    enc = "utf-8"
    with open(input_path, "rb") as fb:
        sig = fb.read(4)
    if sig.startswith(b"\xff\xfe") or sig.startswith(b"\xfe\xff"):
        enc = "utf-16"
    elif sig.startswith(b"\xef\xbb\xbf"):
        enc = "utf-8-sig"

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    cols = [
        "tweet_id", "date", "user_id", "username",
        "followers_count", "friends_count", "statuses_count",
        "raw_content", "reply_count", "retweet_count", "like_count", "quote_count",
        "conversation_id", "hashtags", "mentioned_users", "mentioned_users_ids",
        "view_count", "place", "coordinates",
        "in_reply_to_tweet_id", "in_reply_to_user_id", "in_reply_to_username",
        "has_quote", "quoted_tweet_id", "quoted_user_id", "quoted_username"
    ]

    def extract_hashtags(hts):
        if not isinstance(hts, list):
            return ""
        out = []
        for h in hts:
            if isinstance(h, str):
                out.append(h)
            elif isinstance(h, dict):
                out.append(h.get("name") or h.get("text") or "")
        return ",".join([t for t in out if t])

    def extract_mentioned_users(mus):
        if not isinstance(mus, list):
            return ""
        out = []
        for u in mus:
            if isinstance(u, str):
                out.append(u)
            elif isinstance(u, dict):
                n = u.get("username") or u.get("screen_name") or ""
                if n:
                    out.append(n)
        return ",".join(out)

    def extract_mentioned_user_ids(mus):
        if not isinstance(mus, list):
            return ""
        out = []
        for u in mus:
            if isinstance(u, dict):
                mid = u.get("id")
                if mid is not None:
                    out.append(str(mid))
            elif isinstance(u, (int, str)):
                out.append(str(u))
        return ",".join(out)

    def serialize_obj(obj):
        # Para guardar 'place' o 'coordinates' como JSON en una sola celda
        if obj is None:
            return pd.NA
        if isinstance(obj, (dict, list)):
            return json.dumps(obj, ensure_ascii=False)
        return str(obj)

    rows = []
    with open(input_path, "r", encoding=enc) as f_in:
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            try:
                tw = json.loads(line)
            except json.JSONDecodeError:
                continue

            user = tw.get("user") or {}
            in_reply_user = tw.get("inReplyToUser") or {}
            qt = tw.get("quotedTweet")

            # Manejo minimal de quote:
            has_quote = 0
            quoted_tweet_id = pd.NA
            quoted_user_id = pd.NA
            quoted_username = pd.NA
            if qt is not None:
                has_quote = 1
                if isinstance(qt, dict):
                    quoted_tweet_id = qt.get("id", pd.NA)
                    q_user = qt.get("user") or {}
                    quoted_user_id = q_user.get("id", pd.NA)
                    quoted_username = q_user.get("username", pd.NA)
                elif isinstance(qt, (int, str)):
                    quoted_tweet_id = qt  # si viene solo el id

            row = {
                "tweet_id": tw.get("id"),
                "date": tw.get("date"),
                "user_id": user.get("id"),
                "username": user.get("username"),
                "followers_count": user.get("followersCount"),
                "friends_count": user.get("friendsCount"),
                "statuses_count": user.get("statusesCount"),
                "raw_content": tw.get("rawContent"),
                "reply_count": tw.get("replyCount"),
                "retweet_count": tw.get("retweetCount"),
                "like_count": tw.get("likeCount"),
                "quote_count": tw.get("quoteCount"),
                "conversation_id": tw.get("conversationId"),
                "hashtags": extract_hashtags(tw.get("hashtags")),
                "mentioned_users": extract_mentioned_users(tw.get("mentionedUsers")),
                "mentioned_users_ids": extract_mentioned_user_ids(tw.get("mentionedUsers")),
                "view_count": tw.get("viewCount"),
                "place": serialize_obj(tw.get("place")),
                "coordinates": serialize_obj(tw.get("coordinates")),
                "in_reply_to_tweet_id": tw.get("inReplyToTweetId"),
                "in_reply_to_user_id": in_reply_user.get("id") if isinstance(in_reply_user, dict) else pd.NA,
                "in_reply_to_username": in_reply_user.get("username") if isinstance(in_reply_user, dict) else pd.NA,
                "has_quote": has_quote,
                "quoted_tweet_id": quoted_tweet_id,
                "quoted_user_id": quoted_user_id,
                "quoted_username": quoted_username,
            }

            # Convertir None explícitamente a pd.NA para que se exporte como 'null'
            for k, v in row.items():
                if v is None:
                    row[k] = pd.NA

            rows.append(row)

    df = pd.DataFrame(rows, columns=cols)
    df.to_csv(output_path, index=False, encoding="utf-8", na_rep="null")
    return {"output_csv": output_path, "rows": len(df)}

En el proceso de extracción de datos de los tweets, se seleccionaron 26 columnas del JSON original para preservar en el CSV, priorizando aquellas que contienen información clave para el análisis exploratorio y la construcción de redes.

### Idenficación del tweet

* **tweet_id**: ID único del tweet

  * Sirve para deduplicar, indexar y unir con otras tablas.

* **date**: fecha y hora UTC

  * Sirve para análisis de series temporales (picos por hora/día), relacionar con temporada de lluvias y cambios a lo largo del año.

### Autor (perfil del nodo)

* **user_id, username**: identifican al autor

  * Sirve para construir nodos de la red y etiquetar resultados.

* **followers_count, friends_count, statuses_count**: métricas del autor

  * Sirve para contexto de influencia potencial, comparar engagement relativo (p. ej., likes/followers).

### Contenido (texto y temas)

* **raw_content**: texto original

  * Sirve para tokenización, extracción de toponimia, tópicos y análisis de sentimiento (no mezclar con quoted).

* **hashtags**: lista de temas marcados

  * Sirve para identificar temas frecuentes y co-ocurrencias (wordclouds, top hashtags).

### Interacciones del tweet (engagement)

* **reply_count, retweet_count, like_count, quote_count, view_count**

  * Sirve para medir impacto y alcance, priorizar eventos relevantes y comparar periodos.

### Conversaciones e hilos

* **conversation_id**: agrupa tweets del mismo hilo

  * Sirve para reconstruir discusiones y detectar temas "calientes".

* **in_reply_to_tweet_id, in_reply_to_user_id, in_reply_to_username**: relaciones de respuesta

  * Sirve para construir aristas "reply" en la red (autor → destinatario).

### Red de menciones

* **mentioned_users, mentioned_users_ids**: usuarios mencionados en el texto

  * Sirve para construir aristas "mention" (autor → mencionados), detectar cuentas más aludidas y subredes temáticas.

### Ubicación (cuando exista o se infiera)

* **place, coordinates**: geotag oficial (suele ser null)

  * Sirve para mapeo directo; cuando es null, el `raw_content` permite inferir zonas/avenidas y luego unir a polígonos/centroides.

### Citas (quote)

* **has_quote** (0/1), **quoted_tweet_id, quoted_user_id, quoted_username**

  * Sirve para construir aristas "quote" (autor → citado), identificar fuentes amplificadas y medir centralidad por citas.
  * Nota: el contenido del citado **no** se mezcla con el `raw_content` del autor; se usa solo para la red e influencia.

## Sección 2

In [ ]:
def normalizeText(s: str) -> str:
    if pd.isna(s):
        return s
    s = str(s).lower()
    # quitar saltos de línea
    s = s.replace("\n", " ").replace("\r", " ")
    # quitar urls
    s = URL_RE.sub(" ", s)
    # quitar emojis
    s = EMOJI_RE.sub(" ", s)
    # quitar @ y #
    s = s.replace("@", " ").replace("#", " ")
    # quitar acentos
    s = ''.join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    # normalizar espacios
    s = re.sub(r"\s+", " ", s).strip()
    return s

{'rows_out': 5604, 'output': 'data/clean_traficogt_v2.csv'}


In [ ]:
def applySection2_1() -> pd.DataFrame:
    # cargar csv
    df = pd.read_csv(DATA_CLEAN_IN)

    # aplicar a todas las columnas de texto
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(normalizeText)

    # guardar nuevo csv
    df.to_csv(DATA_CLEAN_OUT, index=False, encoding="utf-8")
    print({"rows_out": len(df), "output": DATA_CLEAN_OUT})
    
    return df

En la limpieza realizada se aplicaron las siguientes transformaciones sobre todas las columnas de texto del archivo:

1. **Conversión a minúsculas**: todo el contenido textual se pasó a minúsculas para uniformar la representación.
2. **Eliminación de saltos de línea**: los caracteres `\n` y `\r` se sustituyeron por espacios, evitando cortes dentro de un mismo campo.
3. **Eliminación de URLs**: se removieron los enlaces que inician con `https://`, evitando ruido en los textos.
4. **Eliminación de emojis**: se quitaron caracteres pertenecientes a los rangos Unicode de símbolos y emojis.
5. **Eliminación de arrobas y numerales**: se suprimieron los símbolos `@` y `#` en cualquier posición del texto.
6. **Eliminación de acentos**: se normalizaron los caracteres con tilde, de modo que `áéíóú` pasaron a `aeiou` y `ñ` a `n`.
7. **Normalización de espacios**: se redujeron múltiples espacios consecutivos a un solo espacio y se recortaron espacios al inicio y al final.

El resultado es un nuevo archivo CSV (`clean_traficogt_v2.csv`) con la misma estructura de columnas, pero con todos los campos de texto **limpios, uniformes y listos para análisis**.

In [51]:
def tokenize(text: str):
    doc = nlp(text)
    return [tok.text for tok in doc if tok.is_alpha]

def lemmatize(text: str):
    doc = nlp(text)
    return [tok.lemma_ for tok in doc if tok.is_alpha and tok.lemma_ not in STOP_ES]

In [ ]:
def applySection2_2(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # normalizar todas las columnas de texto
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(normalizeText)
    
    # features de conteo
    df['exclaim_count']  = df['raw_content'].str.count(r"[!¡]")
    df['question_count'] = df['raw_content'].str.count(r"[?¿]")
    
    # tokenización y lematización
    df['tokens'] = df['raw_content'].apply(tokenize)
    df['lemmas'] = df['raw_content'].apply(lemmatize)
    
    # opcional: guardar CSV
    df.to_csv(DATA_USE, index=False, encoding="utf-8")
    print({"rows_out": len(df), "output": DATA_USE})
    
    return df

{'rows_out': 5604, 'output': 'data/clean_traficogt_v3'}



Después de aplicar esta limpieza y procesamiento de texto, obtendremos un DataFrame enriquecido con nuevas propiedades que permiten un análisis más profundo del contenido textual de los tweets. Específicamente, se generarán:

* **`exclaim_count`**: número de signos de exclamación (`!` o `¡`) en cada tweet, útil para analizar **énfasis o emoción**.
* **`question_count`**: número de signos de interrogación (`?` o `¿`) en cada tweet, útil para detectar **preguntas o curiosidad en el contenido**.
* **`tokens`**: lista de palabras individuales filtradas de caracteres no alfabéticos, que permite análisis de frecuencia, conteo de términos y generación de n-gramas.
* **`lemmas`**: lista de lemas de cada palabra, eliminando stopwords en español, lo que facilita tareas de análisis semántico, agrupación de palabras con la misma raíz y minería de tópicos.

## Sección 3

## Pipeline

In [ ]:
processTweetsToCsv(input_path=DATA_FILE, output_path=DATA_CLEAN)
data_clean1 = applySection2_1()
data_clean2 = applySection2_2(df=data_clean1)